# Text Preprocessing

Processing the text of the airlines dataframe to remove mentions, hashtags, emojis, acronyms etc. to ensure the data is ready for tokenization step

## Importing Packages

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import re
import emoji
import contractions

sns.set_theme(style='ticks')


## Load Dataset

In [ ]:
airlines = pd.read_csv(r'C:\Users\karina mehta\UVA class\Semester 2\Text As Data\us_airlines_dataset\Tweets.csv')
airlines.head(1)


In [ ]:
airline_table = airlines[['airline','retweet_count','text','tweet_created','tweet_location','user_timezone']].copy()

In [ ]:
airline_table.head(1)

## Handling Datatype

In [ ]:
airline_table.dtypes
airline_table['tweet_created'] = pd.to_datetime(airline_table['tweet_created'], utc=True)
airline_table.dtypes

In [ ]:
#Creating Day and Time columns
airline_table['day'] = airline_table['tweet_created'].dt.day
airline_table['hour'] = airline_table['tweet_created'].dt.hour
airline_table['day_of_week'] = airline_table['tweet_created'].dt.dayofweek
airline_table['day_name'] = airline_table['tweet_created'].dt.day_name()
airline_table.head()

## Removing Mentions, URLS and hashtags from text

In [ ]:
def extract_and_clean(text):
    # Extract before removing
    mentions  = re.findall(r'@\w+', text)
    hashtags  = re.findall(r'#\w+', text)

    # Clean text
    text = re.sub(r'@\w+', '', text)                    # remove @mentions
    text = re.sub(r'http\S+|www\S+', '', text)          # remove URLs
    text = re.sub(r'#(\w+)', r'\1', text)               # remove # keep word
    text = re.sub(r'&amp;', '&', text)                  # fix HTML entities
    text = re.sub(r'<.*?>', '', text)                   # fix HTML tags
    text = re.sub(r'\s+', ' ', text)                    # extra whitespace
    text = contractions.fix(text)                       # fix contractions
    text = emoji.demojize(text)                         # convert emojis to text
    text = re.sub(r'(.)\1{2,}', r'\1\1', text)          # remove excessive repetition
    text = text.strip().lower()

    return {
        'cleaned_text' : text,
        'mentions'     : mentions if mentions else None,
        'hashtags'     : hashtags if hashtags else None,
    }

In [ ]:
extracted = airline_table['text'].apply(extract_and_clean)
airline_table['cleaned_text'] = extracted.apply(lambda x: x['cleaned_text'])
airline_table['mentions']     = extracted.apply(lambda x: x['mentions'])
airline_table['hashtags']     = extracted.apply(lambda x: x['hashtags'])

## Expansion

#### Chat Words
Expanding acronyms into full length words

In [ ]:
df_txt  = pd.read_csv('slang/slang.txt', sep='\t', names=['acronym', 'expansion'])  # dilmiter = "\t"


In [ ]:
chat_words = {k.lower(): v.lower() for k, v in df_txt.set_index('acronym')['expansion'].to_dict().items()}

In [ ]:
def chat_conversion(text):
    new_text = []
    for w in text.split():
        if w in chat_words:
            new_text.append(chat_words[w])
        else:
            new_text.append(w)
    return ' '.join(new_text)

In [ ]:
airline_table['cleaned_text'] = airline_table['cleaned_text'].apply(chat_conversion)

In [ ]:
airline_table['text'][3]

In [ ]:
airline_clean = airline_table[['airline','mentions','hashtags','tweet_created','day','hour','day_of_week','day_name','retweet_count','cleaned_text']].copy()
airline_clean.head()

#### Extracting Features

In [ ]:
airline_clean['str_len'] = airline_clean.cleaned_text.str.len()
airline_clean.str_len.describe()

In [ ]:
ax = airline_clean['str_len'].plot.kde()
ax.axvline(1, c='orange', ls='--')
ax.axvline(61, c='orange', ls='--')
ax.axvline(98, c='orange', ls='--')
ax.axvline(122, c='orange', ls='--')
plt.title("Bimodal Distribution of Line Lengths", y=1.05)
sns.despine()
plt.show()

#### Saving Clean Dataset

In [ ]:
airline_clean.head()

In [ ]:
airline_clean.to_parquet('airline_clean.parquet', engine='pyarrow', index=False)